In [14]:
# Import python modules
import os
import sys
import pandas as pd
import numpy as np

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)

In [15]:
# Import custom modules
import plotting
import utils

In [16]:
FOLDER = os.path.join(
    os.path.dirname(os.getcwd()),
    "runs",
    "single_runs",
    "Mar18_Tue_h13-GTSEP_v1_multi",
)
decision_variables_folder = os.path.join(FOLDER, "decision_variables")
model_info_folder = os.path.join(FOLDER, "model_info")

In [17]:
def load_csv_files_from_folder_multi(data_folder_path: str) -> dict[str, pd.DataFrame]:
    if not os.path.exists(data_folder_path):
        raise FileNotFoundError(
            f"{data_folder_path} not found (should be the path to a folder containing processed data in csv files)"
        )
    data = {}
    for file in os.listdir(data_folder_path):
        if file.endswith(".csv"):
            file_path = os.path.join(data_folder_path, file)
            file_name = file.split(".")[0]
            if file_name in [
                "generator_capacity",
                "battery_capacity",
                "branch_capacity",
            ]:
                data[file_name] = pd.read_csv(file_path, index_col=0)
            else:
                data[file_name] = pd.read_csv(file_path, index_col=["year", "hour"])

    return data

In [18]:
# Helper function to reshape a variable with time and other indices
def _reshape_multi(data, index, column_name, value_name):
    """Reshape the data to have time as rows and other index (e.g., generator) as columns."""
    reshaped = data.reset_index().pivot(
        index=index, columns=column_name, values=value_name
    )
    reshaped.columns.name = None  # Remove the name of the columns for cleaner output
    return reshaped

In [19]:
data = load_csv_files_from_folder_multi(decision_variables_folder)

In [20]:
data.keys()

dict_keys(['battery_capacity', 'battery_charging', 'battery_discharging', 'battery_soc', 'branch_capacity', 'curtailment', 'generation', 'generator_capacity', 'load_shedding', 'power_flow'])

# TO-DO:
generator_capacity, battery_capacity, branch_capacity


In [23]:
data["generation"]

ES1 0 CCGT  ES1 0 CCGT new   ES1 0 coal  ES1 0 coal new  \
year hour                                                              
2025 0         0.000000        0.000000   528.956376             0.0   
     1         0.000000        0.000000     9.014363             0.0   
     2         0.000000        0.000000     0.000000             0.0   
     3         0.000000        0.000000     0.000000             0.0   
     4         0.000000        0.000000     0.000000             0.0   
...                 ...             ...          ...             ...   
2045 8755      0.000000    45558.151029  4739.393483             0.0   
     8756      0.000000    37455.926999  4739.393483             0.0   
     8757      0.000000    29604.799967  4739.393483             0.0   
     8758  18991.741279        0.000000  4739.393483             0.0   
     8759  15628.873166        0.000000  4739.393483             0.0   

           ES1 0 offwind-ac  ES1 0 offwind-ac new  ES1 0 onwind  \
year hour                                                         
2025 0          1568.126693                   0.0   5319.767163   
     1          1673.071693                   0.0   4920.986299   
     2          1741.897898                   0.0   4651.677668   
     3          1840.346672                   0.0   4298.965112   
     4          2022.279787                   0.0   4198.939429   
...                     ...                   ...           ...   
2045 8755       1644.757503                   0.0   5879.148404   
     8756       1686.992578                   0.0   6116.501100   
     8757       1767.564016                   0.0   6201.076346   
     8758       2268.769132                   0.0   7349.876700   
     8759       2409.416624                   0.0   7312.760512   

           ES1 0 onwind new  ES1 0 ror  ES1 0 ror new  ...   PT1 0 CCGT  \
year hour                                              ...                
2025 0          8191.786854  32.875547            0.0  ...  4145.000000   
     1          7577.713392  30.824185            0.0  ...  4145.000000   
     2          7163.011239  29.237686            0.0  ...  3125.075762   
     3          6619.877303  27.536529            0.0  ...  2838.315868   
     4          6465.850058  26.418188            0.0  ...  2376.496468   
...                     ...        ...            ...  ...          ...   
2045 8755      20811.461942  44.130818            0.0  ...  4145.000000   
     8756      21651.661283  42.355797            0.0  ...  4145.000000   
     8757      21951.047246  41.136841            0.0  ...  4145.000000   
     8758      26017.659146  40.277869            0.0  ...  4145.000000   
     8759      25886.272406  39.552089            0.0  ...  4145.000000   

           PT1 0 CCGT new  PT1 0 offwind-ac  PT1 0 offwind-ac new  \
year hour                                                           
2025 0        4247.012916        996.684036                   0.0   
     1        4247.012916       1081.533623                   0.0   
     2        4247.012916       1121.634901                   0.0   
     3        4247.012916       1099.962689                   0.0   
     4        4247.012916       1103.704097                   0.0   
...                   ...               ...                   ...   
2045 8755    20827.012916        876.469595                   0.0   
     8756    20827.012916        836.172880                   0.0   
     8757    20827.012916        770.693088                   0.0   
     8758    20827.012916       1064.807020                   0.0   
     8759    20710.567845        854.581462                   0.0   

           PT1 0 onwind  PT1 0 onwind new    PT1 0 ror  PT1 0 ror new  \
year hour                                                               
2025 0       734.029621       1050.763368   286.010867            0.0   
     1       572.628028        819.716996   274.761998            0.0   
     2       530.117812        758.863